# 🌀 Notebook 1 — The Saga Pattern: Why It Exists

You're building an online shop. One **checkout** action must:

1. 📦 reserve stock in the `inventory` service
2. 💳 charge the customer in the `payment` service
3. 🚚 book a courier in the `shipping` service

Each service has **its own database**. They don't share tables, and nothing like `BEGIN TRANSACTION … COMMIT`
can span all three. So what happens if step 2 succeeds but step 3 fails? The customer was charged but
will never receive the item.

This notebook shows:

1. ❌ the **naive** way (no compensation) — and why it corrupts your system
2. ✅ the **saga** way — a sequence of local transactions where each step has an *undo*
3. the vocabulary you'll hear in real designs: *local transaction*, *compensating transaction*,
   *forward* vs *backward* recovery

> **Key idea.** A saga does not *rollback* — it *compensates*. Rollback erases history.
> Compensation adds a new step that reverses the business effect of an earlier step.


## 🛠️ Setup

```bash
cd 05-microservices/saga
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. ❌ Naive approach: call services in sequence, hope for the best

Below we simulate three services with tiny in-memory dictionaries.
The checkout function just calls them one after another. If the last one explodes,
we leave the system in a **broken** state: stock is gone and the customer is charged.


In [1]:
# In-memory "databases" for three services
inventory = {"widget": 10}
payments  = {"total_charged": 0}
shipping  = {"bookings": []}

def reserve_stock(item):
    inventory[item] -= 1
    print(f"  inventory: reserved 1 {item} (left={inventory[item]})")

def charge(amount):
    payments["total_charged"] += amount
    print(f"  payment:   charged ${amount} (total={payments['total_charged']})")

def book_courier(item):
    # 💥 Simulate a crash: courier API is down today
    raise RuntimeError("courier API timeout")

def naive_checkout(item, amount):
    reserve_stock(item)
    charge(amount)
    book_courier(item)     # boom

try:
    naive_checkout("widget", 20)
except Exception as e:
    print(f"✗ checkout failed: {e}")

print()
print("👉 What state are we in now?")
print("   inventory:", inventory)
print("   payments :", payments)
print("   shipping :", shipping)
print()
print("The customer was charged $20 but will NEVER get the widget.")
print("Stock also shows 1 fewer unit even though no order actually completed.")


  inventory: reserved 1 widget (left=9)
  payment:   charged $20 (total=20)
✗ checkout failed: courier API timeout

👉 What state are we in now?
   inventory: {'widget': 9}
   payments : {'total_charged': 20}
   shipping : {'bookings': []}

The customer was charged $20 but will NEVER get the widget.
Stock also shows 1 fewer unit even though no order actually completed.


## 2. ✅ Saga approach: every step has an *undo*

A **saga** is a list of steps, where each step has:

| part | meaning | example |
|---|---|---|
| `do`   | the forward action (a local DB transaction in one service) | `charge($20)` |
| `undo` | the **compensating transaction** that reverses the business effect | `refund($20)` |

When a step fails, we **walk back through the already-completed steps** and run each `undo` in reverse order.
The system ends up in a consistent state — maybe not identical to the start (a refund is visible in
the ledger, for example), but **semantically correct**.

Below is a tiny `Saga` runner. Read it — it's only ~20 lines.


In [2]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Step:
    name: str
    do: Callable[[], None]
    undo: Callable[[], None]

class Saga:
    """Run steps in order. If any step fails, compensate the completed ones in reverse."""
    def __init__(self, steps: list[Step]):
        self.steps = steps

    def run(self) -> str:
        done: list[Step] = []
        for s in self.steps:
            try:
                print(f"→ {s.name}")
                s.do()
                done.append(s)
            except Exception as e:
                print(f"✗ step '{s.name}' failed: {e}")
                for d in reversed(done):
                    print(f"  ↶ compensating '{d.name}'")
                    d.undo()
                return "compensated"
        return "committed"


### Now re-run the e-commerce checkout — this time as a saga

Same three services, same failure in `book_courier`. But now each step has an `undo`,
so when shipping blows up we automatically refund the charge and return stock to inventory.


In [3]:
# Reset the "databases"
inventory = {"widget": 10}
payments  = {"total_charged": 0, "refunds": 0}
shipping  = {"bookings": []}

# --- forward actions ---
def reserve_stock():  inventory["widget"] -= 1
def charge():         payments["total_charged"] += 20
def book_courier():   raise RuntimeError("courier API timeout")  # 💥

# --- compensating actions ---
def release_stock():  inventory["widget"] += 1
def refund():         payments["refunds"] += 20
def cancel_courier(): pass   # nothing to undo — shipping never booked

saga = Saga([
    Step("reserve_stock", reserve_stock, release_stock),
    Step("charge",        charge,        refund),
    Step("book_courier",  book_courier,  cancel_courier),
])

result = saga.run()

print()
print("result   :", result)
print("inventory:", inventory)
print("payments :", payments)
print("shipping :", shipping)
print()
print("✅ Stock is back to 10. The $20 charge was refunded. No phantom order.")


→ reserve_stock
→ charge
→ book_courier
✗ step 'book_courier' failed: courier API timeout
  ↶ compensating 'charge'
  ↶ compensating 'reserve_stock'

result   : compensated
inventory: {'widget': 10}
payments : {'total_charged': 20, 'refunds': 20}
shipping : {'bookings': []}

✅ Stock is back to 10. The $20 charge was refunded. No phantom order.


## 3. Vocabulary you'll see in real designs

- **Local transaction** — a single service's own DB transaction (e.g. `UPDATE inventory …`).
  ACID applies *inside* one service, not across them.
- **Compensating transaction** — a *new* forward transaction that undoes the business effect of a
  completed local transaction. It's recorded in the DB like any other change.
- **Backward recovery** — what we did above: unwind completed steps on failure.
- **Forward recovery** — the opposite: keep retrying the failing step (useful when the failure is transient,
  e.g. a network blip). Real sagas usually **retry first, compensate second**.
- **Pivot transaction** — the point of no return. After this step, going backward is impossible
  (e.g. "send shipment" once the box leaves the warehouse). Before it, compensate; after it, only forward
  recovery is allowed.

### When do I need a saga?

You need one whenever a **single business operation writes to more than one service's database**.
If everything fits in one service, a normal DB transaction is simpler and stronger — don't overthink it.


## ✅ Recap

- Distributed transactions across services **cannot** use classic 2-phase commit at web scale.
- A **saga** is a sequence of local transactions. If any step fails, previous steps are **compensated**.
- Compensation ≠ rollback: it's a new forward action that cancels the earlier business effect.
- Next notebook: **how** to coordinate these steps — with a central brain (orchestration) or by having
  services listen to each other's events (choreography).
